# File Handling

Read and write text, CSV, and JSON files safely using context managers and `pathlib`.

> 📘 **Learning path:** Python Foundations · Notebook 16 of 18

## Learning objectives

- Use context managers to work with files.
- Distinguish read, write, append, and create modes.
- Read files line by line.
- Use Python’s `csv` and `json` modules.
- Build portable paths with `pathlib.Path`.

## How to use this notebook

1. Read each explanation before running its code cell.
2. Run the examples in order and change the values to observe what happens.
3. Complete the practice tasks before opening the suggested solutions at the end.

**Requirements:** Python 3.10+ and a Jupyter-compatible environment. The examples use the Python standard library unless stated otherwise.

## 1. Read and write files safely

Use `open()` or the `pathlib.Path` methods to work with files. A `with` statement is recommended because it closes the file automatically, including when an error occurs. Specify `encoding="utf-8"` for text files unless you have a reason to use another encoding.

## 2. Write and read a text file

The following example uses a temporary directory so the exercise leaves no extra file in the repository. In your own project, replace the temporary path with a deliberate location.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as temp_dir:
    path = Path(temp_dir) / "notes.txt"
    path.write_text("First line\nSecond line\n", encoding="utf-8")
    content = path.read_text(encoding="utf-8")
    print(content, end="")
    print("Line count:", len(content.splitlines()))

First line
Second line
Line count: 2


## 3. File modes

Common modes for `open()` include:

- `"r"`: read an existing file.
- `"w"`: write, replacing existing contents or creating a file.
- `"a"`: append to an existing file or create it.
- `"x"`: create a new file and fail if it already exists.
- Add `"b"` for binary mode, such as `"rb"`.

Be careful with `"w"`: it truncates a file when it opens successfully.

In [2]:
with TemporaryDirectory() as temp_dir:
    path = Path(temp_dir) / "log.txt"
    path.write_text("start\n", encoding="utf-8")
    with path.open("a", encoding="utf-8") as file:
        file.write("process completed\n")
    print(path.read_text(encoding="utf-8"))

start
process completed



## 4. Process lines without reading everything first

For large text files, iterating over the file object can avoid loading every line into memory at once. Choose `read_text()` when the file is small and reading it as a whole is convenient.

In [3]:
with TemporaryDirectory() as temp_dir:
    path = Path(temp_dir) / "values.txt"
    path.write_text("12\n8\n15\n", encoding="utf-8")
    values = []
    with path.open("r", encoding="utf-8") as file:
        for line in file:
            stripped = line.strip()
            if stripped:
                values.append(int(stripped))
    print(values)
    print("Total:", sum(values))

[12, 8, 15]
Total: 35


## 5. Work with CSV files

CSV files store tabular data as text. Python's built-in `csv` module handles quoting and delimiters more reliably than manually splitting each line on commas.

In [4]:
import csv

with TemporaryDirectory() as temp_dir:
    path = Path(temp_dir) / "sites.csv"
    rows = [
        {"site": "A", "use": "park", "area": 1200},
        {"site": "B", "use": "clinic", "area": 800},
    ]
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["site", "use", "area"])
        writer.writeheader()
        writer.writerows(rows)
    with path.open("r", newline="", encoding="utf-8") as file:
        for row in csv.DictReader(file):
            print(row)

{'site': 'A', 'use': 'park', 'area': '1200'}
{'site': 'B', 'use': 'clinic', 'area': '800'}


## 6. Work with JSON files

JSON is a common format for structured data. The `json` module can serialize dictionaries and lists to JSON and parse JSON back into Python objects. Use `ensure_ascii=False` when you want non-ASCII text to remain readable in the file.

In [5]:
import json

record = {"name": "Sample Park", "open": True, "features": ["trees", "walkway"]}
with TemporaryDirectory() as temp_dir:
    path = Path(temp_dir) / "record.json"
    path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    loaded = json.loads(path.read_text(encoding="utf-8"))
    print(loaded["name"])
    print(loaded["features"])

Sample Park
['trees', 'walkway']


## 7. File paths with `pathlib`

`Path` provides a platform-aware way to join and inspect paths. Useful methods include `.name`, `.suffix`, `.exists()`, `.parent`, `.mkdir()`, `.read_text()`, and `.write_text()`. A relative path is interpreted relative to the current working directory.

In [6]:
path = Path("data") / "measurements.csv"
print("Path:", path)
print("Name:", path.name)
print("Suffix:", path.suffix)
print("Parent:", path.parent)

Path: data/measurements.csv
Name: measurements.csv
Suffix: .csv
Parent: data


## Common mistakes and safe habits

- Always close files; a `with` block does this for you.
- Specify text encoding explicitly for portable text processing.
- Avoid accidentally overwriting important data with write mode.
- CSV should be parsed with `csv`, not a basic `split(',')`, because fields may contain commas or quotes.
- Never execute or trust an untrusted serialized payload just because it was read from a file.

## 7. Understand file modes and path checks

- `"r"` reads an existing file.
- `"w"` creates or truncates a file.
- `"a"` appends to a file.
- `"x"` creates a new file and fails if it already exists.

Use `Path.exists()` for a basic check, but still handle errors when opening files because files can change between checking and use.

In [7]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as temp_dir:
    path = Path(temp_dir) / "report.txt"
    print(path.exists())
    with path.open("x", encoding="utf-8") as file:
        file.write("First version\n")
    print(path.exists(), path.read_text(encoding="utf-8").strip())

False
True First version


## 8. Handle missing files and choose an encoding

Use `with` so the file is closed even when an error occurs. `encoding="utf-8"` makes text encoding explicit. A missing file usually raises `FileNotFoundError`; the right response depends on whether absence is expected.

In [8]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as temp_dir:
    missing = Path(temp_dir) / "missing.txt"
    try:
        missing.read_text(encoding="utf-8")
    except FileNotFoundError:
        print("No saved file exists yet.")

No saved file exists yet.


## 9. Read and write JSON

JSON is a common format for configuration and structured records. Use `json.dump()`/`json.load()` with file objects or `json.dumps()`/`json.loads()` with strings. JSON supports objects, arrays, strings, numbers, booleans, and `null`; it does not directly represent every Python type.

In [9]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

record = {"name": "Central Park", "active": True, "visitors": 1200}
with TemporaryDirectory() as temp_dir:
    path = Path(temp_dir) / "record.json"
    path.write_text(json.dumps(record, indent=2), encoding="utf-8")
    restored = json.loads(path.read_text(encoding="utf-8"))
    print(restored["name"], restored["active"])

Central Park True


## 10. CSV and newline handling

When using the `csv` module, opening the file with `newline=""` allows the CSV module to handle newline conventions correctly. Always be explicit about encoding when reading text data from external sources.

In [10]:
import csv
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as temp_dir:
    path = Path(temp_dir) / "counts.csv"
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["place", "count"])
        writer.writeheader()
        writer.writerow({"place": "Central", "count": 12})
    with path.open("r", newline="", encoding="utf-8") as file:
        print(next(csv.DictReader(file)))

{'place': 'Central', 'count': '12'}


## 🧩 Practice tasks

Try to solve these tasks without looking at the solutions. Write your attempts in new code cells below each task.

1. Write three lines to a text file in a temporary directory, then read and print them.
2. Create a CSV file with `name` and `score` columns and read it with `csv.DictReader`.
3. Save a nested dictionary as JSON and load it back.
4. Explain why `with open(...)` is preferable to manually opening and closing a file.

## ✅ Suggested solutions

These are sample solutions, not the only correct answers. Try the exercises first, then compare your approach.

In [11]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as temp_dir:
    path = Path(temp_dir) / "notes.txt"
    path.write_text("First line\nSecond line\n", encoding="utf-8")
    content = path.read_text(encoding="utf-8")
    print(content, end="")
    print("Line count:", len(content.splitlines()))

First line
Second line
Line count: 2


## Key takeaways

- Focus on understanding what the code does, not memorizing every example.
- Make small changes and predict the result before running the cell.
- When an error appears, read its type and message; debugging is part of learning.

**Next notebook:** Exception Handling.